# PufferDrive on Colab

This notebook trains PufferDrive on a Colab T4. It does two things:

1. Sets up the repo so a restart does not mean a 20 minute rebuild. The setup
   cell reconnects to Drive, clones the repo, and caches compiled extensions and
   converted maps.
2. Trains the reward-conditioning branch of PufferDrive, where each agent gets
   its own reward weights. The last cells run a sweep on a frozen checkpoint to
   see whether those weights change behaviour.

The setup cell handles three cases:

1. Kernel restart or short disconnect: the VM is still alive, so it checks a
   marker and exits in about five seconds.
2. Fresh VM with a warm Drive cache: shallow clone, restore compiled extensions
   and maps from Drive, 4 to 6 minutes.
3. First run: clone, build, download and convert the dataset, 15 to 25 minutes.

The clone uses my fork (`alexanderdbondar/PufferDrive`, branch
`reward-conditioning`). Change `REPO_URL` or `REPO_BRANCH` in the setup cell for
other targets. Compiled extensions are cached per commit, so changes to C code
trigger a rebuild.

The Drive cache (`MyDrive/pufferdrive_cache/`) holds the converted `.bin` maps,
not the raw JSON, so it stays under about 15 GB. The one-time pipeline downloads
the zip, extracts the JSONs, converts them with `drive.py`, caches the bins, and
deletes the JSONs.


In [ ]:
import os, sys, shutil, subprocess, glob

CACHE      = "/content/drive/MyDrive/pufferdrive_cache"
REPO       = "/content/PufferDrive"
MARKER     = f"{REPO}/.setup_complete"
DATA_REPO  = "daphne-cornelisse/pufferdrive_womd_train"
REPO_URL   = "https://github.com/alexanderdbondar/PufferDrive.git"
# fork branch with reward conditioning; use upstream and "2.0" for vanilla
REPO_BRANCH = "reward-conditioning"

def sh(cmd):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"Command failed: {cmd}")
    return r.stdout

def git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True)

def imports_ok():
    """Check whether this VM has a working setup at the branch head."""
    if not (os.path.isdir(REPO) and os.path.exists(MARKER)):
        return False
    r = subprocess.run(
        [sys.executable, "-c", "import pufferlib._C, pufferlib.ocean.drive.drive"],
        cwd=REPO, capture_output=True, text=True)
    if r.returncode != 0:
        return False
    if git("rev-parse", "--abbrev-ref", "HEAD").stdout.strip() != REPO_BRANCH:
        return False
    if git("remote", "get-url", "origin").stdout.strip() != REPO_URL:
        return False
    if git("fetch", "origin", REPO_BRANCH).returncode != 0:
        return False  # offline: keep what we have instead of wiping it
    head = git("rev-parse", "HEAD").stdout.strip()
    target = git("rev-parse", f"origin/{REPO_BRANCH}").stdout.strip()
    return bool(head) and head == target

# Fast path: restart or short disconnect on the same VM
if imports_ok():
    print("PufferDrive is ready, nothing to do.")
else:
    # 1. Mount Drive
    if not os.path.ismount("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    for d in ("repo", "data", "build", "pip"):
        os.makedirs(f"{CACHE}/{d}", exist_ok=True)
    os.environ["PIP_CACHE_DIR"] = f"{CACHE}/pip"

    # 2. Clone the repo. It always comes from git, not Drive: the repo has
    # symlinks that Drive cannot store, and a shallow clone is cheap.
    if os.path.isdir(REPO) and not os.path.isdir(f"{REPO}/.git"):
        print("Removing partial/broken repo directory...")
        shutil.rmtree(REPO)
    if not os.path.isdir(REPO):
        print("Cloning PufferDrive (shallow)...")
        sh(f"git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO}")
    os.chdir(REPO)

    # reset any checkout left by a restart to the branch head
    sh(f"git remote set-url origin {REPO_URL}")
    sh(f"git fetch origin {REPO_BRANCH}")
    sh(f"git checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}")

    # 3. Install PyTorch if it is missing
    try:
        import torch
        need_torch = not torch.cuda.is_available()
    except ImportError:
        need_torch = True
    if need_torch:
        print("Installing PyTorch (CUDA)...")
        sh(f"{sys.executable} -m pip install --no-cache-dir -q "
           f"torch torchvision torchaudio "
           f"--index-url https://download.pytorch.org/whl/cu121")
    else:
        print(f"PyTorch OK ({torch.__version__}, CUDA available), skipping reinstall.")

    # 4. Restore compiled extensions or rebuild if the commit changed
    def so_paths():
        return glob.glob("pufferlib/**/*.so", recursive=True)

    # the cache key includes the git hash, so new C code does not load old .so files
    git_hash = sh("git rev-parse --short HEAD").strip()
    bcache = f"{CACHE}/build/py{sys.version_info.major}{sys.version_info.minor}-{git_hash}"
    if not so_paths() and os.path.isdir(bcache):
        print("Restoring compiled extensions from Drive...")
        sh(f"cp -a {bcache}/. .")

    r = subprocess.run([sys.executable, "-c", "import pufferlib._C, pufferlib.ocean.drive.drive"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("Building PufferDrive (deps + C extensions)...")
        # --no-deps avoids pip building sdist dependencies (meson and others)
        # that fail on newer Python versions. setup.py falls back to CPU if nvcc
        # is missing.
        sh(f"{sys.executable} -m pip install --no-deps --no-build-isolation -q .")
        sh(f"{sys.executable} -m pip install -q gymnasium ale-py shimmy "
           f"huggingface_hub wandb rich_argparse pyro-ppl tensorboard "
           f'"heavyball<2.0.0"')  # Muon optimizer (default.ini)
        sh(f"{sys.executable} setup.py build_ext --inplace")
    shutil.rmtree(bcache, ignore_errors=True)
    os.makedirs(bcache, exist_ok=True)
    for so in so_paths():
        dst = os.path.join(bcache, so)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(so, dst)

    # 5. Convert the dataset once and cache only the .bin maps
    # HF zip -> data/processed/training/*.json -> drive.py -> resources/.../map_XXX.bin
    real_bins = "pufferlib/resources/drive/binaries/training"
    if not os.path.lexists(real_bins):
        os.makedirs("pufferlib/resources/drive/binaries", exist_ok=True)
        if glob.glob(f"{CACHE}/data/binaries/training/*.bin"):
            print("Linking converted .bin maps from Drive cache...")
        else:
            if not glob.glob("data/processed/training/*.json"):
                print("Downloading WOMD dataset (one-time, ~4.8 GB zip)...")
                sh(f"hf download {DATA_REPO} --repo-type dataset --local-dir /content/_dl")
                sh("unzip -q -o /content/_dl/training.zip -d /content/_dl/")
                src = "/content/_dl/data/processed"
                if not os.path.isdir(src):
                    src = "/content/_dl/processed"
                shutil.move(src, f"{REPO}/data/processed")
                shutil.rmtree("/content/_dl", ignore_errors=True)  # keep the zip off Drive
            print("Converting JSON scenes -> compact .bin maps...")
            sh(f"{sys.executable} pufferlib/ocean/drive/drive.py")
            shutil.move(real_bins, f"{CACHE}/data/binaries/training")  # cache the bins, delete the JSONs
            shutil.rmtree(f"{REPO}/data", ignore_errors=True)
        os.symlink(f"{CACHE}/data/binaries/training", real_bins)

    # 6. Done
    shutil.rmtree(f"{CACHE}/repo", ignore_errors=True)  # remove the old cache layout
    with open(MARKER, "w") as f:
        f.write("ok")
    print("\nSetup complete.")


## Train

The next two cells write the training script and start it. The config is sized
for the T4 (16 GB VRAM, 2 vCPU): a small CPU rollout stack, a larger training
batch, and checkpoints on Drive so a disconnect only costs a few minutes.


In [ ]:
# Flags: edit this cell, rerun it, then run the next cell
USE_WANDB = True  # Weights & Biases logging
USE_CONDITIONING = False  # per-agent reward weights
USE_RESUME = False  # continue from the latest checkpoint saved on Drive
WANDB_PROJECT = "pufferdrive"
WANDB_GROUP = "colab-t4"

train_script = """
import os
import sys
import time
import torch
import pufferlib
import pufferlib.pufferl

USE_WANDB = {use_wandb!r}
USE_CONDITIONING = {use_conditioning!r}
USE_RESUME = {use_resume!r}
WANDB_PROJECT = {project!r}
WANDB_GROUP = {group!r}


def patch_wandb_compat():
    # newer wandb versions removed wandb.util.generate_id, which pufferl still uses
    import wandb.util

    if not hasattr(wandb.util, "generate_id"):
        from wandb.sdk.lib import runid

        wandb.util.generate_id = runid.generate_id


class DashboardFilter:
    # hide PufferLib's ANSI dashboard so Colab output stays readable
    def __init__(self, original_stdout):
        self.terminal = original_stdout

    def write(self, message):
        if "0;0H" in message or "\u256d\u2500\u2500\u2500" in message or "\u2502  PufferLib" in message:
            return
        self.terminal.write(message)

    def flush(self):
        self.terminal.flush()


if __name__ == "__main__":
    sys.stdout = DashboardFilter(sys.stdout)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {{device}}")

    env_name = "puffer_drive"
    dict_args = pufferlib.pufferl.load_config(env_name)

    dict_args["vec"]["num_workers"] = 1
    dict_args["vec"]["num_envs"] = 4
    dict_args["vec"]["zero_copy"] = False
    dict_args["vec"]["batch_size"] = 4

    dict_args["env"]["num_maps"] = 2000

    dict_args["env"]["conditioned"] = int(USE_CONDITIONING)
    # override knob ranges here; the full list is in pufferlib/config/ocean/drive.ini
    # dict_args["env"]["cond_collision_min"] = -1.5

    if USE_RESUME:
        import glob
        _pattern = "/content/drive/MyDrive/pufferdrive_cache/runs/**/model_*.pt"
        _hits = glob.glob(_pattern, recursive=True)
        if _hits:
            _latest = max(_hits, key=os.path.getmtime)
            print(f"Resuming from checkpoint: {{_latest}}")
            dict_args["load_model_path"] = _latest
        else:
            print("No checkpoint on Drive yet, starting fresh")

    dict_args["train"]["device"] = device
    dict_args["train"]["rollout_horizon"] = 32
    # num_envs * num_agents = 4 * 1024, must fit batch_size // rollout_horizon
    dict_args["train"]["batch_size"] = 131072  # 4096 * 32
    dict_args["train"]["minibatch_size"] = 16384  # host RAM (~12.7 GB), not VRAM, is the limit here
    dict_args["train"]["max_minibatch_size"] = 16384

    # keep checkpoints on Drive so a disconnect loses minutes, not the run
    import time as _time
    _ckpt_dir = "/content/drive/MyDrive/pufferdrive_cache/runs/" + _time.strftime("%m%d_%H%M%S")
    dict_args["train"]["data_dir"] = _ckpt_dir
    print("Checkpoints -> " + _ckpt_dir)

    # Weights & Biases. pufferl also accepts --wandb and --wandb-project.
    if USE_WANDB:
        try:  # read the key from Colab secrets, otherwise log in interactively
            from google.colab import userdata
            os.environ.setdefault("WANDB_API_KEY", userdata.get("WANDB_API_KEY"))
        except Exception:
            pass
        import wandb
        patch_wandb_compat()
        wandb.login()
        dict_args["wandb"] = True
        dict_args["wandb_project"] = WANDB_PROJECT
        dict_args["wandb_group"] = WANDB_GROUP
        dict_args["wandb_name"] = f"t4_{{time.strftime('%m%d_%H%M%S')}}"
    else:
        dict_args["wandb"] = False

    print("Starting pufferl.train()...")
    pufferlib.pufferl.train(
        env_name=env_name,
        args=dict_args,
        vecenv=None,
        policy=None,
    )
""".format(
    use_wandb=USE_WANDB,
    use_conditioning=USE_CONDITIONING,
    use_resume=USE_RESUME,
    project=WANDB_PROJECT,
    group=WANDB_GROUP,
)

with open("run_colab_train.py", "w") as f:
    f.write(train_script)

print("Training script written (run_colab_train.py).")


In [ ]:
print("Starting training session...")
!python run_colab_train.py

## Conditioning sweep: does it work?

Freeze a trained policy and replay it with all agents pinned to one allowed knob
value at a time (a degenerate `min == max` range). If conditioning works, the
collision rate should fall as crash aversion rises.

Run this after training, when checkpoints are in
`MyDrive/pufferdrive_cache/runs/<timestamp>/`. Pass the newest
`model_puffer_drive_*.pt` with `--checkpoint`, or use `--find-latest`.


In [ ]:
# sweep_conditioning.py is in the fork's reward-conditioning branch.
# Pin every agent to the same collision penalty:
!python sweep_conditioning.py --find-latest --knob collision --values 0 -0.375 -0.75 -1.125 -1.5 --episodes 4 --agents 128 --out-prefix conditioning_sweep_collision

# another knob:
# !python sweep_conditioning.py --knob goal_radius --values 2 5 8 12 --episodes 4


## Sanity check (optional)

Render a few episodes without a display to check the environment.


In [ ]:
!python pufferlib/ocean/drive/drive.py